# Stage 2 — Adaptive Risk Matching & Pricing Optimizer

Estimate local realized default risk and rival price pressure for each Batch 3 applicant, then choose the interest rate maximizing expected profit across the three equally likely borrower-preference types.

### Key Enhancements:
1. **Dynamic 10% Risk Cut:** Instead of an arbitrary 40% risk ceiling, the kernel-estimated local default risk is evaluated across all applicants, and the top 10% riskiest borrowers are rejected.
2. **Applicant-Specific Adaptive Rate Grid:** Instead of a fixed 3%–25% grid, candidate rates for each borrower are generated dynamically between the min and max competitor quotes of their matching peers (allowing rates to scale up to ~95%+ to match market reality).
3. **Controlled Profit Creaming:** The minimum expected profit threshold is calibrated to cream at most 1% of the non-risky applicant pool.
4. **Rich Diagnostic Plots:** Competitor pricing distributions over risk deciles, risk cutoff distributions, and expected profit distributions.

In [ ]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PRINCIPAL = 10_000.0
TASTE_DISCOUNT = 0.02
RISK_CUT_PERCENTILE = 0.90     # Cut the 10% most likely to default
PROFIT_CREAM_PERCENTILE = 0.01 # Cream at most 1% on expected profit
DEFAULT_PROFIT_FLOOR = 5.0     # Baseline floor (~1% of positive EV)
N_RATES = 60                   # Granularity of adaptive candidate rate grid
H_GRID = [0.005, 0.010, 0.015, 0.025]
STRESS_DELTAS = [-0.01, 0.01]
WEIGHT_CUTOFF = 1e-4
CHUNK_SIZE = 1000
print('Configuration initialized with adaptive grid and dynamic risk cutoff.')

## 1. Load and validate inputs

Join applicant-level predicted PDs and diagnostic values on unique IDs. Confirm coverage and expose the limitations caused by missing rival quotes.

In [ ]:
batch2 = pd.read_csv(PROCESSED_DIR / 'Batch2.csv')
batch2_pd = pd.read_csv(PROCESSED_DIR / 'batch2_p_default_catboost.csv')
batch3 = pd.read_csv(RAW_DIR / 'NewApplications_Lender3_Round2.csv')
batch3_pd = pd.read_csv(PROCESSED_DIR / 'batch3_p_default_catboost.csv')
diag = pd.read_csv(RAW_DIR / 'Round1_Diagnostic_33.csv')

for name, frame in [('Batch2', batch2), ('Batch2 PD', batch2_pd), ('Batch3', batch3), ('Batch3 PD', batch3_pd), ('diagnostic', diag)]:
    assert frame.id.is_unique, f'{name} IDs are not unique'
assert all(len(x) == 100_000 for x in [batch2, batch2_pd, batch3, batch3_pd, diag])

# Merge predictions: Batch2.csv already contains the diagnostic columns (default, accepted, profit, etc.)
hist = batch2.merge(batch2_pd[['id', 'pd']].rename(columns={'pd': 'p_hist'}), on='id', validate='one_to_one')
target = batch3.merge(batch3_pd[['id', 'pd']].rename(columns={'pd': 'p_target'}), on='id', validate='one_to_one')

assert hist.p_hist.between(0, 1).all() and target.p_target.between(0, 1).all()
assert hist.default.isin([0, 1]).all() and hist.accepted.isin([0, 1]).all()
comp_mask = hist[['competing1', 'competing2']].notna().all(axis=1).to_numpy()

print(f'Batch 2: {len(hist):,} labeled; complete rival quotes: {comp_mask.sum():,} ({comp_mask.mean():.1%})')
print(f'Observed accepted/won rate={hist.accepted.mean():.2%}; default among accepted={hist.loc[hist.accepted.eq(1), "default"].mean():.2%}')
print(f'Observed Stage 1 diagnostic profit: €{hist.profit.sum():,.0f}')

## 2. Kernel risk and market-choice functions

Use normalized Gaussian weights `exp(-0.5*((p_query-p_ref)/h)^2)`. A KD-tree radius includes all observations with raw weight ≥1e-4; excluded weights are smaller. Risk is estimated over all labeled Batch 2 cases; competitive probabilities use local neighbors among complete rival quotes, with their weights renormalized. Strict undercut conditions match the user-provided specification.

In [ ]:
def build_tree(reference_pd, h, cutoff=WEIGHT_CUTOFF):
    ref = np.asarray(reference_pd, dtype=float)
    tree = cKDTree(ref.reshape(-1, 1))
    radius = h * math.sqrt(2 * math.log(1 / cutoff))
    return ref, tree, radius

def kernel_neighbors(query_pd, ref_pd, tree, radius, h):
    q = np.asarray(query_pd, dtype=float)
    groups = tree.query_ball_point(q.reshape(-1, 1), r=radius)
    result = []
    for qi, inds in zip(q, groups):
        inds = np.asarray(inds, dtype=np.int32)
        w = np.exp(-0.5 * ((qi - ref_pd[inds]) / h) ** 2)
        total = w.sum()
        if total <= 0 or not np.isfinite(total):
            inds = np.array([int(np.argmin(np.abs(ref_pd - qi)))], dtype=np.int32)
            w = np.ones(1); total = 1.0
        result.append((inds, w / total))
    return result

def get_adaptive_rate_grid(rival1, rival2, n_points=N_RATES, buffer=0.01):
    """Construct an applicant-specific rate grid between local min and max competitor quotes."""
    all_rivals = np.r_[rival1, rival2]
    all_rivals = all_rivals[np.isfinite(all_rivals)]
    if len(all_rivals) == 0:
        return np.linspace(0.03, 0.25, n_points)
    r_min = max(0.005, float(np.min(all_rivals)) - buffer)
    r_max = min(1.000, float(np.max(all_rivals)) + buffer)
    if r_max <= r_min:
        r_max = r_min + 0.05
    return np.linspace(r_min, r_max, n_points)

def scenario_pwin_exact(rate, rival1, rival2, delta=0.0):
    """P(Win) for one specific rival pair under equal borrower preference types."""
    c1 = float(rival1) * (1.0 + delta)
    c2 = float(rival2) * (1.0 + delta)
    r = np.asarray(rate, dtype=float)
    win1 = (r < c1 - TASTE_DISCOUNT) & (r < c2)
    win2 = (r < c1) & (r < c2 - TASTE_DISCOUNT)
    win3 = (r - TASTE_DISCOUNT < c1) & (r - TASTE_DISCOUNT < c2)
    return (win1.astype(float) + win2.astype(float) + win3.astype(float)) / 3.0

def expected_profit(rate_grid, p_local, p_win):
    """P(Win) * €10,000 * [(1 - p_local) * rate - p_local]."""
    r = np.asarray(rate_grid, dtype=float)
    return np.asarray(p_win) * PRINCIPAL * ((1.0 - p_local) * r - p_local)

def p_win_by_rate(rate_grid, rival1, rival2, weights, delta=0.0):
    """Mean P(Win) over lender-1, lender-2, and lender-3 preference types for a group of rivals."""
    c1 = np.asarray(rival1, dtype=float) * (1.0 + delta)
    c2 = np.asarray(rival2, dtype=float) * (1.0 + delta)
    r = np.asarray(rate_grid, dtype=float)[:, None]
    win1 = ((r < c1[None, :] - TASTE_DISCOUNT) & (r < c2[None, :])).astype(float)
    win2 = ((r < c1[None, :]) & (r < c2[None, :] - TASTE_DISCOUNT)).astype(float)
    win3 = ((r - TASTE_DISCOUNT < c1[None, :]) & (r - TASTE_DISCOUNT < c2[None, :])).astype(float)
    w = np.asarray(weights, dtype=float); w = w / w.sum()
    return ((win1 @ w) + (win2 @ w) + (win3 @ w)) / 3.0

## 3. Exploratory Analysis: Rival Pricing & Rate Distribution over Risk

Examine how Competitor 1 and Competitor 2 price loans as a function of credit risk in Batch 2. Notice that rival rates range well above 25% (up to ~95%), motivating the applicant-specific adaptive rate grid.

In [ ]:
# Plot distribution of competing1 and competing2 over P(Default) in Batch 2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hist_valid = hist.loc[comp_mask].copy()
hist_valid['risk_decile'] = pd.qcut(hist_valid['p_hist'], 10)
decile_summary = hist_valid.groupby('risk_decile', observed=False).agg(
    p_mean=('p_hist', 'mean'),
    c1_median=('competing1', 'median'),
    c1_p10=('competing1', lambda x: np.quantile(x, 0.10)),
    c1_p90=('competing1', lambda x: np.quantile(x, 0.90)),
    c2_median=('competing2', 'median'),
    c2_p10=('competing2', lambda x: np.quantile(x, 0.10)),
    c2_p90=('competing2', lambda x: np.quantile(x, 0.90))
)

ax = axes[0]
ax.plot(decile_summary.p_mean, decile_summary.c1_median, marker='o', label='Competitor 1 Median', color='#0072B2', lw=2)
ax.fill_between(decile_summary.p_mean, decile_summary.c1_p10, decile_summary.c1_p90, color='#0072B2', alpha=0.15, label='Comp 1 [10%-90%]')
ax.plot(decile_summary.p_mean, decile_summary.c2_median, marker='s', label='Competitor 2 Median', color='#E69F00', lw=2)
ax.fill_between(decile_summary.p_mean, decile_summary.c2_p10, decile_summary.c2_p90, color='#E69F00', alpha=0.15, label='Comp 2 [10%-90%]')
ax.set_title('Rival Rates vs Credit Risk (Batch 2)', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Probability of Default')
ax.set_ylabel('Quoted Rate')
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
ax.hist(hist_valid['competing1'], bins=40, alpha=0.5, label='Competitor 1', color='#0072B2', density=True)
ax.hist(hist_valid['competing2'], bins=40, alpha=0.5, label='Competitor 2', color='#E69F00', density=True)
ax.axvline(0.25, color='red', linestyle='--', label='Old Static Cap (25%)', lw=2)
ax.set_title('Rival Rate Distributions (Tails extend to ~95%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Interest Rate')
ax.set_ylabel('Density')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f'Rival quotes > 25%: Competitor 1={(hist.competing1 > 0.25).sum():,} | Competitor 2={(hist.competing2 > 0.25).sum():,}')
print(f'Max rates: Competitor 1={hist.competing1.max():.2%} | Competitor 2={hist.competing2.max():.2%}')

## 4. Batch 2 Backtest: 10% Risk Cut & Adaptive Rate Grid

For each query applicant in the complete-quote cohort:
1. Compute leave-one-out local default risk using the KD-tree.
2. Determine the 90th percentile of local risk and **reject the 10% riskiest applicants**.
3. For accepted applicants, build an adaptive rate grid between local min and max rival quotes, and select the rate maximizing expected profit.

In [ ]:
hist_pd = hist.p_hist.to_numpy(float)
hist_default = hist.default.to_numpy(int)
hist_c1 = hist.competing1.to_numpy(float)
hist_c2 = hist.competing2.to_numpy(float)
complete_idx = np.flatnonzero(comp_mask)
bt = hist.iloc[complete_idx].reset_index(drop=True)
bt_pd = hist_pd[complete_idx]
bt_default = hist_default[complete_idx]
bt_c1 = hist_c1[complete_idx]
bt_c2 = hist_c2[complete_idx]

def run_backtest(h, delta=0.0, show_progress=True):
    ref, tree, radius = build_tree(hist_pd, h)
    bids = np.full(len(bt), np.nan)
    local = np.full(len(bt), np.nan)
    maxev = np.full(len(bt), np.nan)
    pwin_best = np.full(len(bt), np.nan)
    
    # Precompute leave-one-out local risk
    iterator = range(0, len(bt), CHUNK_SIZE)
    if show_progress:
        iterator = tqdm(iterator, total=math.ceil(len(bt) / CHUNK_SIZE), desc=f'Backtest h={h:.3f}, delta={delta:+.0%}', unit='1%')
    
    cached_matches = []
    for start in iterator:
        stop = min(start + CHUNK_SIZE, len(bt))
        nb = kernel_neighbors(bt_pd[start:stop], ref, tree, radius, h)
        for k, (inds, w) in enumerate(nb):
            row = start + k
            query_global = complete_idx[row]
            keep = inds != query_global
            inds = inds[keep]; w = w[keep]
            if len(inds):
                w_norm = w / w.sum()
                local[row] = float(w_norm @ hist_default[inds])
                cached_matches.append((row, inds, w))
            else:
                cached_matches.append((row, np.empty(0, dtype=int), np.empty(0, dtype=float)))
    
    # Dynamic 10% risk cutoff (cut the 10% most likely to default)
    valid_risk = local[np.isfinite(local)]
    risk_cutoff = float(np.quantile(valid_risk, RISK_CUT_PERCENTILE))
    
    # Optimize rates with adaptive grid
    for row, inds, w in cached_matches:
        if np.isnan(local[row]) or local[row] > risk_cutoff:
            continue  # Reject top 10% risk
        
        # Local competitor quotes
        cm = comp_mask[inds]
        cinds = inds[cm]
        if len(cinds) == 0:
            continue
        
        rates = get_adaptive_rate_grid(hist_c1[cinds], hist_c2[cinds], n_points=N_RATES)
        pw = scenario_pwin_exact(rates, bt_c1[row], bt_c2[row], delta=delta)
        ev = expected_profit(rates, local[row], pw)
        j = int(np.argmax(ev))
        maxev[row] = ev[j]
        pwin_best[row] = pw[j]
        if ev[j] >= DEFAULT_PROFIT_FLOOR:
            bids[row] = rates[j]
            
    c1 = bt_c1 * (1.0 + delta); c2 = bt_c2 * (1.0 + delta); typ = bt.borrowertype.to_numpy(int)
    own_effective = bids.copy(); own_effective[typ == 3] -= TASTE_DISCOUNT
    rival1_effective = c1.copy(); rival1_effective[typ == 1] -= TASTE_DISCOUNT
    rival2_effective = c2.copy(); rival2_effective[typ == 2] -= TASTE_DISCOUNT
    won = np.isfinite(bids) & (own_effective < rival1_effective) & (own_effective < rival2_effective)
    realized = np.where(won, PRINCIPAL * ((1.0 - bt_default) * bids - bt_default), 0.0)
    d_rate = float(bt_default[won].mean()) if won.any() else np.nan
    return {'h': h, 'delta': delta, 'risk_cutoff': risk_cutoff, 'n_quote_complete': len(bt),
            'n_bids': int(np.isfinite(bids).sum()), 'n_won': int(won.sum()),
            'win_rate_among_complete': float(won.mean()), 'default_rate_won': d_rate,
            'total_profit_eur': float(realized.sum()), 'mean_rate_won': float(bids[won].mean()) if won.any() else np.nan,
            'bids': bids, 'local_risk': local, 'expected_profit': maxev, 'p_win_at_optimum': pwin_best, 'won': won}

backtests = []
for h in H_GRID:
    result = run_backtest(h)
    backtests.append(result)
    print({k: v for k, v in result.items() if k not in {'bids', 'local_risk', 'expected_profit', 'p_win_at_optimum', 'won'}})
backtest_summary = pd.DataFrame([{k: v for k, v in r.items() if k not in {'bids', 'local_risk', 'expected_profit', 'p_win_at_optimum', 'won'}} for r in backtests])
display(backtest_summary)

## 5. Select bandwidth and stress-test competitor rates

Choose the bandwidth with the largest realized Batch 2 replay P&L. Then replay at ±1% relative competitor-rate shifts. These sensitivity cases alter rivals' observed quotes but preserve all other assumptions.

In [ ]:
best_idx=int(backtest_summary.total_profit_eur.idxmax())
SELECTED_H=float(backtest_summary.loc[best_idx,'h'])
selected_backtest=backtests[best_idx]
print(f'Selected h={SELECTED_H:.3f}; backtest P&L €{selected_backtest["total_profit_eur"]:,.0f}; won-loan default rate {selected_backtest["default_rate_won"]:.2%}')
stress=[]
for delta in STRESS_DELTAS:
    r=run_backtest(SELECTED_H,delta=delta)
    stress.append({k:v for k,v in r.items() if k not in {'bids','local_risk','expected_profit','p_win_at_optimum','won'}})
stress_summary=pd.DataFrame(stress)
display(stress_summary)

## 6. Optimize Batch 3 Rates with Adaptive Pricing & 10% Risk Cut

1. Estimate local default risk for all 100,000 Batch 3 applicants using the Batch 2 KD-tree.
2. Determine the 90th percentile risk threshold and reject the top 10% riskiest applicants.
3. For accepted applicants, construct the applicant-specific candidate rate grid between local rival quotes.
4. Dynamically set the profit floor to cream at most 1% of the non-risky applicants.

In [ ]:
def price_batch3(query_pd, h, show_progress=True):
    ref, tree, radius = build_tree(hist_pd, h)
    q = np.asarray(query_pd, dtype=float)
    n = len(q)
    out_rate = np.full(n, np.nan)
    out_local = np.full(n, np.nan)
    out_ev = np.full(n, np.nan)
    out_pw = np.full(n, np.nan)
    
    chunks = range(0, n, CHUNK_SIZE)
    iterator = tqdm(chunks, total=math.ceil(n / CHUNK_SIZE), desc='Batch 3 Applicants', unit='1%') if show_progress else chunks
    
    cached_queries = []
    for start in iterator:
        stop = min(start + CHUNK_SIZE, n)
        for offset, (inds, w) in enumerate(kernel_neighbors(q[start:stop], ref, tree, radius, h)):
            i = start + offset
            out_local[i] = float(w @ hist_default[inds])
            cached_queries.append((i, inds, w))
            
    # Dynamic 10% risk cutoff (cut 10% most likely to default)
    risk_cutoff_b3 = float(np.quantile(out_local, RISK_CUT_PERCENTILE))
    print(f'Batch 3 90th percentile risk cutoff: {risk_cutoff_b3:.2%} (rejecting {n * (1 - RISK_CUT_PERCENTILE):,.0f} riskiest applicants)')
    
    # Compute raw optimal bids without profit floor first to calibrate the 1% creaming threshold
    raw_evs = []
    optim_data = []
    for i, inds, w in cached_queries:
        if out_local[i] > risk_cutoff_b3:
            continue
        cm = comp_mask[inds]
        cinds = inds[cm]
        if not len(cinds):
            continue
        cw = w[cm]; cw = cw / cw.sum()
        rates = get_adaptive_rate_grid(hist_c1[cinds], hist_c2[cinds], n_points=N_RATES)
        pw = p_win_by_rate(rates, hist_c1[cinds], hist_c2[cinds], cw)
        ev = expected_profit(rates, out_local[i], pw)
        best = int(np.argmax(ev))
        raw_evs.append(ev[best])
        optim_data.append((i, rates[best], ev[best], pw[best]))
        
    # Calibrate profit floor to cream at most 1% of positive-EV applicants
    positive_evs = [v for v in raw_evs if v > 0]
    calibrated_floor = max(0.0, float(np.quantile(positive_evs, PROFIT_CREAM_PERCENTILE))) if len(positive_evs) else DEFAULT_PROFIT_FLOOR
    print(f'Calibrated profit floor (creaming 1% lowest positive EV): €{calibrated_floor:.2f}')
    
    for i, best_rate, ev, pw in optim_data:
        out_ev[i] = ev
        out_pw[i] = pw
        if ev >= calibrated_floor:
            out_rate[i] = best_rate
            
    df_res = pd.DataFrame({'rate': out_rate, 'p_local': out_local, 'max_expected_profit': out_ev, 'p_win_at_optimum': out_pw})
    return df_res, risk_cutoff_b3, calibrated_floor

batch3_result, b3_risk_cutoff, b3_profit_floor = price_batch3(target.p_target.to_numpy(), SELECTED_H, show_progress=True)
final_bids = pd.DataFrame({
    'id': target.id.to_numpy(),
    'rate': batch3_result.rate.to_numpy(),
    'p_default': target.p_target.to_numpy(),
    'p_local': batch3_result.p_local.to_numpy(),
    'max_expected_profit': batch3_result.max_expected_profit.to_numpy(),
    'p_win_at_optimum': batch3_result.p_win_at_optimum.to_numpy()
})
assert len(final_bids) == 100_000 and final_bids.id.is_unique
n_risk_reject = int((final_bids.p_local > b3_risk_cutoff).sum())
n_profit_reject = int((final_bids.rate.isna() & (final_bids.p_local <= b3_risk_cutoff)).sum())
print(f'Final bids issued: {final_bids.rate.notna().sum():,} | Risk rejects (top 10%): {n_risk_reject:,} | Profit/market rejects: {n_profit_reject:,}')
print(f'Mean issued rate: {final_bids.rate.mean():.2%} | Mean expected profit among bids: €{final_bids.loc[final_bids.rate.notna(), "max_expected_profit"].mean():,.2f}')
display(final_bids.head())

In [ ]:
# Diagnostic Figures: Risk Cutoff, Expected Profit Distribution & Final Adaptive Rates
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Local Risk Distribution & 10% Cutoff
ax = axes[0]
ax.hist(final_bids['p_local'], bins=50, color='#0072B2', alpha=0.6, label='Retained (90%)')
ax.hist(final_bids.loc[final_bids['p_local'] > b3_risk_cutoff, 'p_local'], bins=20, color='red', alpha=0.7, label='Cut Top 10%')
ax.axvline(b3_risk_cutoff, color='darkred', linestyle='--', lw=2, label=f'Cutoff = {b3_risk_cutoff:.1%}')
ax.set_title('Batch 3 Local Risk & 10% Cutoff', fontweight='bold')
ax.set_xlabel('Kernel Local Default Risk')
ax.set_ylabel('Applicant Count')
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Expected Profit Distribution & 1% Creaming Floor
ax = axes[1]
valid_evs = final_bids.loc[final_bids['p_local'] <= b3_risk_cutoff, 'max_expected_profit'].dropna()
ax.hist(valid_evs[valid_evs.between(0, 500)], bins=50, color='#009E73', alpha=0.6)
ax.axvline(b3_profit_floor, color='darkred', linestyle='--', lw=2, label=f'Floor = €{b3_profit_floor:.2f} (cream 1%)')
ax.axvline(50.0, color='orange', linestyle=':', lw=2, label='Old Floor = €50')
ax.set_title('Expected Profit Distribution (Non-Risky Cohort)', fontweight='bold')
ax.set_xlabel('Max Expected Profit (€)')
ax.set_ylabel('Applicant Count')
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 3: Final Issued Rates vs Default Risk
ax = axes[2]
bids_made = final_bids.dropna(subset=['rate'])
ax.scatter(bids_made['p_default'], bids_made['rate'], alpha=0.10, s=5, color='#CC79A7')
ax.set_title('Adaptive Bids vs Default Risk (Batch 3)', fontweight='bold')
ax.set_xlabel('Model Predicted PD')
ax.set_ylabel('Issued Rate')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Check benchmark gates and export

The required deliverable is a two-column CSV with `id, rate`; rejected applicants remain present with missing rates for auditability. A second detailed file retains the inputs and optimization diagnostics. Export a concise backtest and stress-test summary as well.

In [ ]:
TARGET_PROFIT_EUR=30_900_000
TARGET_DEFAULT_RATE=0.1225
bt_profit=float(selected_backtest['total_profit_eur'])
bt_default=float(selected_backtest['default_rate_won']) if np.isfinite(selected_backtest['default_rate_won']) else np.nan
gate_report=pd.DataFrame([
 {'gate':'Backtest P&L exceeds €30.9M','observed':bt_profit,'threshold':TARGET_PROFIT_EUR,'pass':bt_profit>TARGET_PROFIT_EUR},
 {'gate':'Won-loan default rate below 12.25%','observed':bt_default,'threshold':TARGET_DEFAULT_RATE,'pass':np.isfinite(bt_default) and bt_default<TARGET_DEFAULT_RATE},
])
gate_report['sample_n']=[selected_backtest['n_quote_complete'],selected_backtest['n_won']]
display(gate_report)
print(f"Benchmark covers {selected_backtest['n_quote_complete']:,} of 100,000 Batch 2 cases with complete rival quotes.")

final_bids[['id','rate']].to_csv(PROCESSED_DIR/'batch3_lender3_rates_stage2.csv',index=False)
final_bids.to_csv(PROCESSED_DIR/'batch3_lender3_rates_stage2_audit.csv',index=False)
backtest_summary.to_csv(PROCESSED_DIR/'stage2_bandwidth_backtest.csv',index=False)
stress_summary.to_csv(PROCESSED_DIR/'stage2_competitor_stress_test.csv',index=False)
gate_report.to_csv(PROCESSED_DIR/'stage2_backtest_gates.csv',index=False)
print('Wrote Stage 2 bid CSV, audit, bandwidth, stress-test, and gate outputs.')